# quick version

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import pdist
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

# =========================================================
# paths
# =========================================================
NEURAL_CSV = "/root/reveal_roi_state_space_results/reveal_level_with_roi_state_scores.csv"
RSA_MODEL_CSV = "/root/bayesian_belief_table_results/rsa_model_input_table.csv"
OUT_DIR = "/root/acc_rsa_results_fast_1"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================================================
# settings
# =========================================================
KEY_COLS = ["Subject", "Game", "reveal_idx"]
OPTIONAL_KEY_COLS = ["condition_label"]

ACC_STATE_COLS = ["acc_PC1", "acc_PC2", "acc_PC3"]

# 只保留主变量，先快速出结果
MODEL_VARS_MAIN = [
    "uncertainty_imbalance_before",
    "bestdeck_entropy_before",
    "belief_surprise_kl",
    "value_gap_before",
]

MODEL_VARS = MODEL_VARS_MAIN

# 快速版参数
N_PERM = 200
RANDOM_STATE = 42
MIN_TRIALS_PER_SUBJECT = 6
MIN_SUBJECTS_PER_BLOCK = 3
LATE_WINDOW = [4, 5, 6]

# =========================================================
# helpers
# =========================================================
def is_binary_series(s):
    s = pd.Series(s).dropna()
    vals = pd.unique(s)
    if len(vals) <= 2:
        return True
    try:
        vals_num = pd.to_numeric(vals)
        if set(vals_num).issubset({0, 1}):
            return True
    except Exception:
        pass
    return False


def is_categorical_series(s):
    s = pd.Series(s).dropna()
    if s.dtype == object or str(s.dtype).startswith("category"):
        return True
    if is_binary_series(s):
        return True
    return False


def zscore_safe(x):
    x = np.asarray(x, dtype=float)
    sd = np.nanstd(x)
    if sd < 1e-12:
        return np.zeros_like(x, dtype=float)
    return (x - np.nanmean(x)) / sd


def neural_rdm_vector(X, metric="euclidean"):
    return pdist(np.asarray(X, dtype=float), metric=metric)


def model_rdm_vector(values, categorical=False, normalize_continuous=True):
    s = pd.Series(values)

    if categorical:
        x = s.astype(str).to_numpy()
        n = len(x)
        out = []
        for i in range(n - 1):
            for j in range(i + 1, n):
                out.append(float(x[i] != x[j]))
        return np.array(out, dtype=float)

    x = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
    if normalize_continuous:
        x = zscore_safe(x)

    n = len(x)
    out = []
    for i in range(n - 1):
        for j in range(i + 1, n):
            out.append(abs(x[i] - x[j]))
    return np.array(out, dtype=float)


def compute_rsa_single_subject(df_sub, state_cols, var, categorical):
    df_sub = df_sub.dropna(subset=state_cols + [var]).copy()
    if len(df_sub) < MIN_TRIALS_PER_SUBJECT:
        return np.nan, 0

    X = df_sub[state_cols].to_numpy(dtype=float)
    neural_vec = neural_rdm_vector(X, metric="euclidean")
    model_vec = model_rdm_vector(df_sub[var], categorical=categorical, normalize_continuous=True)

    valid = np.isfinite(neural_vec) & np.isfinite(model_vec)
    neural_vec = neural_vec[valid]
    model_vec = model_vec[valid]

    if len(neural_vec) < 5:
        return np.nan, len(neural_vec)

    if np.nanstd(neural_vec) < 1e-12 or np.nanstd(model_vec) < 1e-12:
        return np.nan, len(neural_vec)

    rho, _ = spearmanr(neural_vec, model_vec)
    return float(rho), int(len(neural_vec))


def permute_var_within_subject(df_in, var, rng):
    out = df_in.copy().reset_index(drop=True)
    vals = out[var].to_numpy(copy=True)

    for subj, idx in out.groupby("Subject").groups.items():
        idx = np.array(list(idx), dtype=int)
        vals[idx] = rng.permutation(vals[idx])

    out[var] = vals
    return out


def subjectwise_observed_rhos(df_block, state_cols, var, categorical):
    df_block = df_block.reset_index(drop=True).copy()

    rows = []
    for subj, g in df_block.groupby("Subject"):
        g = g.reset_index(drop=True).copy()
        rho, n_pairs = compute_rsa_single_subject(g, state_cols, var, categorical)
        rows.append({
            "Subject": subj,
            "rho": rho,
            "n_pairs": n_pairs,
            "n_trials": len(g),
        })
    out = pd.DataFrame(rows)
    out = out[np.isfinite(out["rho"])].copy()
    return out


def permutation_test_subjectwise_mean(df_block, state_cols, var, categorical,
                                      n_perm=200, random_state=42):
    rng = np.random.default_rng(random_state)

    obs_df = subjectwise_observed_rhos(df_block, state_cols, var, categorical)
    if len(obs_df) < MIN_SUBJECTS_PER_BLOCK:
        return np.nan, np.nan, 0, 0

    obs_mean = obs_df["rho"].mean()
    n_subjects_used = obs_df["Subject"].nunique()
    n_pairs_used = obs_df["n_pairs"].sum()

    perm_means = []
    for _ in range(n_perm):
        perm_df = permute_var_within_subject(df_block, var, rng)
        tmp = subjectwise_observed_rhos(perm_df, state_cols, var, categorical)
        if len(tmp) < MIN_SUBJECTS_PER_BLOCK:
            perm_means.append(np.nan)
        else:
            perm_means.append(tmp["rho"].mean())

    perm_means = np.array(perm_means, dtype=float)
    perm_means = perm_means[np.isfinite(perm_means)]

    if len(perm_means) == 0:
        return float(obs_mean), np.nan, int(n_subjects_used), int(n_pairs_used)

    pval = (np.sum(np.abs(perm_means) >= abs(obs_mean)) + 1) / (len(perm_means) + 1)
    return float(obs_mean), float(pval), int(n_subjects_used), int(n_pairs_used)


def run_rsa_block_fast(df_block, block_name, model_vars, out_dir, state_cols=ACC_STATE_COLS,
                       n_perm=200, random_state=42):
    rows = []

    needed = state_cols + ["Subject", "Game", "reveal_idx"]
    df_block = df_block.dropna(subset=needed).copy()

    for var in model_vars:
        if var not in df_block.columns:
            continue

        sub = df_block.dropna(subset=[var] + state_cols).copy()
        if len(sub) < 10:
            rows.append({
                "block": block_name,
                "model_var": var,
                "rsa_rho": np.nan,
                "perm_p": np.nan,
                "n_trials": len(sub),
                "n_subjects": sub["Subject"].nunique(),
                "n_pairs_used": 0,
                "var_type": "missing_or_too_small",
            })
            continue

        categorical = is_categorical_series(sub[var])

        rho, pval, n_subjects_used, n_pairs_used = permutation_test_subjectwise_mean(
            sub,
            state_cols=state_cols,
            var=var,
            categorical=categorical,
            n_perm=n_perm,
            random_state=random_state,
        )

        rows.append({
            "block": block_name,
            "model_var": var,
            "rsa_rho": rho,
            "perm_p": pval,
            "n_trials": len(sub),
            "n_subjects": n_subjects_used,
            "n_pairs_used": n_pairs_used,
            "var_type": "categorical" if categorical else "continuous",
        })

    out_df = pd.DataFrame(rows).sort_values(["perm_p", "rsa_rho"], na_position="last")
    out_df.to_csv(os.path.join(out_dir, f"rsa_{block_name}.csv"), index=False)
    return out_df


def plot_rsa_by_reveal(res_df, outpath, title):
    plt.figure(figsize=(8, 5))
    for var in MODEL_VARS_MAIN:
        sub = res_df[res_df["model_var"] == var].sort_values("reveal_idx")
        if len(sub) == 0:
            continue
        plt.plot(sub["reveal_idx"], sub["rsa_rho"], marker="o", label=var)

    plt.axhline(0, linestyle="--", linewidth=1)
    plt.axvline(4, linestyle=":", linewidth=1)
    plt.xlabel("Reveal index")
    plt.ylabel("Mean subject-wise RSA (rho)")
    plt.title(title)
    plt.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()


# =========================================================
# load and merge
# =========================================================
neural_df = pd.read_csv(NEURAL_CSV)
model_df = pd.read_csv(RSA_MODEL_CSV)

print("Loaded neural:", neural_df.shape)
print("Loaded model :", model_df.shape)

merge_keys = KEY_COLS.copy()
for c in OPTIONAL_KEY_COLS:
    if c in neural_df.columns and c in model_df.columns:
        merge_keys.append(c)

neural_extra_cols = []
if "has_acc_data" in neural_df.columns:
    neural_extra_cols.append("has_acc_data")

model_keep_cols = merge_keys + [v for v in MODEL_VARS if v in model_df.columns]
model_keep_cols = list(dict.fromkeys(model_keep_cols))

neural_keep_cols = merge_keys + neural_extra_cols + [c for c in ACC_STATE_COLS if c in neural_df.columns]
neural_keep_cols = list(dict.fromkeys(neural_keep_cols))

df = neural_df[neural_keep_cols].merge(
    model_df[model_keep_cols],
    on=merge_keys,
    how="inner"
)

required_after_merge = merge_keys + ACC_STATE_COLS
missing_req = [c for c in required_after_merge if c not in df.columns]
if missing_req:
    raise ValueError(f"Missing required columns after merge: {missing_req}")

if "has_acc_data" in df.columns:
    df = df[df["has_acc_data"] == 1].copy()

df = df.dropna(subset=ACC_STATE_COLS).copy()
df = df.sort_values(["Subject", "Game", "reveal_idx"]).reset_index(drop=True)
df.to_csv(os.path.join(OUT_DIR, "acc_rsa_merged_table_fast.csv"), index=False)

print("Merged shape:", df.shape)
print("Subjects:", df["Subject"].nunique())
print("Games   :", df["Game"].nunique())

# =========================================================
# reveal-wise RSA
# =========================================================
reveal_rows = []

for reveal in sorted(df["reveal_idx"].dropna().unique()):
    block_df = df[df["reveal_idx"] == reveal].copy()
    block_name = f"reveal_{int(reveal)}"

    out_df = run_rsa_block_fast(
        block_df,
        block_name=block_name,
        model_vars=MODEL_VARS,
        out_dir=OUT_DIR,
        state_cols=ACC_STATE_COLS,
        n_perm=N_PERM,
        random_state=RANDOM_STATE,
    )
    out_df["reveal_idx"] = reveal
    reveal_rows.append(out_df)

reveal_rsa_df = pd.concat(reveal_rows, axis=0, ignore_index=True)
reveal_rsa_df.to_csv(os.path.join(OUT_DIR, "acc_revealwise_rsa_summary_fast.csv"), index=False)

# =========================================================
# reveal 4 focused RSA
# =========================================================
if 4 in df["reveal_idx"].unique():
    reveal4_df = df[df["reveal_idx"] == 4].copy()
    rsa_r4 = run_rsa_block_fast(
        reveal4_df,
        block_name="reveal4_focus_fast",
        model_vars=MODEL_VARS,
        out_dir=OUT_DIR,
        state_cols=ACC_STATE_COLS,
        n_perm=N_PERM,
        random_state=RANDOM_STATE,
    )
else:
    rsa_r4 = pd.DataFrame()

# =========================================================
# late pooled RSA (4-6)
# =========================================================
late_df = df[df["reveal_idx"].isin(LATE_WINDOW)].copy()
rsa_late = run_rsa_block_fast(
    late_df,
    block_name="late46_pooled_fast",
    model_vars=MODEL_VARS,
    out_dir=OUT_DIR,
    state_cols=ACC_STATE_COLS,
    n_perm=N_PERM,
    random_state=RANDOM_STATE,
)

# =========================================================
# equal / unequal split RSA (late window)
# =========================================================
split_rows = []
if "condition_label" in late_df.columns:
    for cond in sorted(late_df["condition_label"].dropna().unique()):
        cond_df = late_df[late_df["condition_label"] == cond].copy()
        tmp = run_rsa_block_fast(
            cond_df,
            block_name=f"late46_{cond}_fast",
            model_vars=MODEL_VARS,
            out_dir=OUT_DIR,
            state_cols=ACC_STATE_COLS,
            n_perm=N_PERM,
            random_state=RANDOM_STATE,
        )
        tmp["condition_split"] = cond
        split_rows.append(tmp)

if len(split_rows) > 0:
    split_df = pd.concat(split_rows, axis=0, ignore_index=True)
    split_df.to_csv(os.path.join(OUT_DIR, "acc_late46_condition_split_rsa_fast.csv"), index=False)
else:
    split_df = pd.DataFrame()

# =========================================================
# plots
# =========================================================
plot_rsa_by_reveal(
    reveal_rsa_df,
    outpath=os.path.join(OUT_DIR, "acc_revealwise_rsa_main_fast.png"),
    title="ACC reveal-wise RSA (fast subject-wise)"
)

if len(rsa_r4) > 0:
    rr = rsa_r4.copy().sort_values("rsa_rho", ascending=False)
    plt.figure(figsize=(7, 5))
    plt.barh(rr["model_var"], rr["rsa_rho"])
    plt.axvline(0, linestyle="--", linewidth=1)
    plt.xlabel("Mean subject-wise RSA (rho)")
    plt.title("ACC RSA at reveal 4 (fast)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "acc_reveal4_rsa_bar_fast.png"), dpi=200)
    plt.close()

if len(rsa_late) > 0:
    rr = rsa_late.copy().sort_values("rsa_rho", ascending=False)
    plt.figure(figsize=(7, 5))
    plt.barh(rr["model_var"], rr["rsa_rho"])
    plt.axvline(0, linestyle="--", linewidth=1)
    plt.xlabel("Mean subject-wise RSA (rho)")
    plt.title("ACC RSA in late window 4-6 (fast)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "acc_late46_rsa_bar_fast.png"), dpi=200)
    plt.close()

# =========================================================
# summary
# =========================================================
summary = []
summary.append("=== merged table ===")
summary.append(str(df.shape))
summary.append(f"n_subjects = {df['Subject'].nunique()}")
summary.append(f"n_games = {df['Game'].nunique()}")

summary.append("\n=== reveal-wise results ===")
for reveal in sorted(reveal_rsa_df["reveal_idx"].dropna().unique()):
    sub = (
        reveal_rsa_df[reveal_rsa_df["reveal_idx"] == reveal]
        .sort_values(["perm_p", "rsa_rho"], na_position="last")
    )
    summary.append(f"\nReveal {int(reveal)}")
    summary.append(sub.to_string(index=False))

if len(rsa_r4) > 0:
    summary.append("\n=== reveal 4 focus ===")
    summary.append(rsa_r4.sort_values(["perm_p", "rsa_rho"], na_position="last").to_string(index=False))

summary.append("\n=== late pooled (4-6) ===")
summary.append(rsa_late.sort_values(["perm_p", "rsa_rho"], na_position="last").to_string(index=False))

if len(split_df) > 0:
    summary.append("\n=== late pooled split by condition ===")
    for cond in sorted(split_df["condition_split"].dropna().unique()):
        summary.append(f"\nCondition = {cond}")
        summary.append(
            split_df[split_df["condition_split"] == cond]
            .sort_values(["perm_p", "rsa_rho"], na_position="last")
            .to_string(index=False)
        )

with open(os.path.join(OUT_DIR, "acc_rsa_summary_fast.txt"), "w") as f:
    f.write("\n".join(summary))

print("\nDone.")
print("Saved to:", OUT_DIR)

# current version for ACC (2 metrics)

# one-side

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl

from scipy.spatial.distance import pdist
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

# =========================================================
# paths & settings
# =========================================================
MERGED_CSV = "/root/acc_rsa_results_fast/acc_rsa_merged_table_fast.csv"
OUT_DIR = "/root/acc_rsa_results_final_one_sided"
os.makedirs(OUT_DIR, exist_ok=True)

ACC_STATE_COLS = ["acc_PC1", "acc_PC2", "acc_PC3"]

FOCUS_VARS = [
    "bestdeck_entropy_before",
    "uncertainty_imbalance_before",
]

# 建议边界 p 值结果用更多 permutation 稳定估计
N_PERM = 5000
N_BOOT = 2000
RANDOM_STATE_BASE = 4

MIN_TRIALS_PER_SUBJECT = 6
MIN_SUBJECTS_PER_BLOCK = 5

EARLY_WINDOW = [1, 2, 3]
LATE_WINDOW = [4, 5, 6]

# 固定不同 block 的随机种子，避免 Python hash 随会话变化
BLOCK_SEEDS = {
    "early13": RANDOM_STATE_BASE + 101,
    "late46": RANDOM_STATE_BASE + 202,
}

# =========================================================
# Nature-style config
# =========================================================
def set_nature_style():
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "font.size": 7,
        "axes.labelsize": 7.5,
        "axes.titlesize": 8.5,
        "xtick.labelsize": 6.5,
        "ytick.labelsize": 6.5,
        "legend.fontsize": 6.5,
        "axes.linewidth": 0.8,
        "lines.linewidth": 1.6,
        "figure.dpi": 150,
        "savefig.dpi": 600,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
    })

set_nature_style()

# =========================================================
# helper functions
# =========================================================
def is_binary_series(s):
    s = pd.Series(s).dropna()
    vals = pd.unique(s)
    if len(vals) <= 2:
        return True
    try:
        vals_num = pd.to_numeric(vals)
        if set(vals_num).issubset({0, 1}):
            return True
    except Exception:
        pass
    return False


def is_categorical_series(s):
    s = pd.Series(s).dropna()
    if len(s) == 0:
        return True
    if s.dtype == object or str(s.dtype).startswith("category"):
        return True
    if is_binary_series(s):
        return True
    return False


def zscore_safe(x):
    x = np.asarray(x, dtype=float)
    sd = np.nanstd(x)
    if sd < 1e-12:
        return np.zeros_like(x, dtype=float)
    return (x - np.nanmean(x)) / sd


def neural_rdm_vector(X):
    return pdist(np.asarray(X, dtype=float), metric="euclidean")


def model_rdm_vector(values, categorical=False, normalize_continuous=True):
    s = pd.Series(values)

    if categorical:
        x = s.astype(str).to_numpy()
        n = len(x)
        out = [
            float(x[i] != x[j])
            for i in range(n - 1)
            for j in range(i + 1, n)
        ]
        return np.array(out, dtype=float)

    x = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
    if normalize_continuous:
        x = zscore_safe(x)

    n = len(x)
    out = [
        abs(x[i] - x[j])
        for i in range(n - 1)
        for j in range(i + 1, n)
    ]
    return np.array(out, dtype=float)


def compute_rsa_single_subject(df_sub, state_cols, var, categorical):
    df_sub = (
        df_sub
        .dropna(subset=state_cols + [var])
        .copy()
        .reset_index(drop=True)
    )

    if len(df_sub) < MIN_TRIALS_PER_SUBJECT:
        return np.nan, 0

    X = df_sub[state_cols].to_numpy(dtype=float)
    neural_vec = neural_rdm_vector(X)
    model_vec = model_rdm_vector(df_sub[var], categorical=categorical)

    valid = np.isfinite(neural_vec) & np.isfinite(model_vec)
    neural_vec = neural_vec[valid]
    model_vec = model_vec[valid]

    if (
        len(neural_vec) < 5
        or np.nanstd(neural_vec) < 1e-12
        or np.nanstd(model_vec) < 1e-12
    ):
        return np.nan, len(neural_vec)

    rho, _ = spearmanr(neural_vec, model_vec)
    return float(rho), int(len(neural_vec))


def subjectwise_observed_rhos(df_block, state_cols, var, categorical):
    rows = []

    for subj, g in df_block.groupby("Subject"):
        g = g.reset_index(drop=True).copy()
        rho, n_pairs = compute_rsa_single_subject(
            g,
            state_cols,
            var,
            categorical,
        )
        rows.append({
            "Subject": subj,
            "rho": rho,
            "n_pairs": n_pairs,
            "n_trials": len(g),
        })

    out = pd.DataFrame(rows)
    return out[np.isfinite(out["rho"])].copy()


def permute_var_within_subject(df_in, var, rng):
    out = df_in.copy().reset_index(drop=True)
    vals = out[var].to_numpy(copy=True)

    for subj, idx in out.groupby("Subject").groups.items():
        idx = np.array(list(idx), dtype=int)
        vals[idx] = rng.permutation(vals[idx])

    out[var] = vals
    return out


def permutation_test_subjectwise_mean_one_sided_positive(
    df_block,
    state_cols,
    var,
    categorical,
    n_perm=50000,
    random_state=42,
):
    """
    One-sided positive RSA permutation test.

    H1: observed mean RSA rho > expected mean RSA rho under the null.

    p = (1 + number of permuted means >= observed mean) / (1 + number of permutations)
    """
    rng = np.random.default_rng(random_state)

    obs_df = subjectwise_observed_rhos(
        df_block,
        state_cols,
        var,
        categorical,
    )

    if len(obs_df) < MIN_SUBJECTS_PER_BLOCK:
        return np.nan, np.nan, 0, 0, obs_df

    obs_mean = obs_df["rho"].mean()
    n_subjects_used = obs_df["Subject"].nunique()
    n_pairs_used = obs_df["n_pairs"].sum()

    perm_means = []

    for _ in range(n_perm):
        perm_df = permute_var_within_subject(df_block, var, rng)
        tmp = subjectwise_observed_rhos(
            perm_df,
            state_cols,
            var,
            categorical,
        )

        if len(tmp) >= MIN_SUBJECTS_PER_BLOCK:
            perm_means.append(tmp["rho"].mean())

    perm_means = np.array(perm_means, dtype=float)
    perm_means = perm_means[np.isfinite(perm_means)]

    if len(perm_means) == 0:
        return (
            float(obs_mean),
            np.nan,
            int(n_subjects_used),
            int(n_pairs_used),
            obs_df,
        )

    perm_p_one_sided_positive = (
        np.sum(perm_means >= obs_mean) + 1
    ) / (len(perm_means) + 1)

    return (
        float(obs_mean),
        float(perm_p_one_sided_positive),
        int(n_subjects_used),
        int(n_pairs_used),
        obs_df,
    )


def bootstrap_ci_subjectwise_mean(
    df_block,
    state_cols,
    var,
    categorical,
    n_boot=10000,
    random_state=42,
):
    rng = np.random.default_rng(random_state)

    obs_df = subjectwise_observed_rhos(
        df_block,
        state_cols,
        var,
        categorical,
    )

    if len(obs_df) < MIN_SUBJECTS_PER_BLOCK:
        return np.nan, np.nan

    subs = obs_df["Subject"].unique()
    subj_to_rho = dict(zip(obs_df["Subject"], obs_df["rho"]))

    boots = []

    for _ in range(n_boot):
        samp_subs = rng.choice(subs, size=len(subs), replace=True)
        samp_rhos = [
            subj_to_rho[s]
            for s in samp_subs
            if np.isfinite(subj_to_rho.get(s, np.nan))
        ]

        if len(samp_rhos) >= MIN_SUBJECTS_PER_BLOCK:
            boots.append(np.mean(samp_rhos))

    boots = np.array(boots, dtype=float)
    boots = boots[np.isfinite(boots)]

    if len(boots) == 0:
        return np.nan, np.nan

    lo = np.percentile(boots, 2.5)
    hi = np.percentile(boots, 97.5)

    return float(lo), float(hi)


def run_rsa_block_one_sided(
    df_block,
    block_name,
    model_vars,
    out_dir,
    state_cols=ACC_STATE_COLS,
):
    rows = []
    subj_detail_rows = []

    df_block = (
        df_block
        .dropna(subset=state_cols + ["Subject", "Game", "reveal_idx"])
        .copy()
        .reset_index(drop=True)
    )

    block_seed = BLOCK_SEEDS.get(block_name, RANDOM_STATE_BASE + 999)

    print(f"\n=== Running RSA block: {block_name} ===")
    print(
        f"Trials: {len(df_block)}, "
        f"Subjects: {df_block['Subject'].nunique()}, "
        f"Reveals: {sorted(df_block['reveal_idx'].dropna().unique())}"
    )

    for vi, var in enumerate(model_vars):
        if var not in df_block.columns:
            print(f"⚠ Skip {var}: column not found.")
            continue

        sub = (
            df_block
            .dropna(subset=[var] + state_cols)
            .copy()
            .reset_index(drop=True)
        )

        if len(sub) < 10:
            print(f"⚠ Skip {var}: too few valid rows ({len(sub)}).")
            continue

        categorical = is_categorical_series(sub[var])
        var_seed = block_seed + vi * 1000

        print(
            f"  - Variable: {var} | "
            f"type: {'categorical' if categorical else 'continuous'} | "
            f"valid trials: {len(sub)}"
        )

        rho, p_one_pos, n_subjects_used, n_pairs_used, obs_df = (
            permutation_test_subjectwise_mean_one_sided_positive(
                sub,
                state_cols,
                var,
                categorical,
                n_perm=N_PERM,
                random_state=var_seed,
            )
        )

        ci_lo, ci_hi = bootstrap_ci_subjectwise_mean(
            sub,
            state_cols,
            var,
            categorical,
            n_boot=N_BOOT,
            random_state=var_seed + 123,
        )

        rows.append({
            "block": block_name,
            "model_var": var,
            "rsa_rho": rho,
            "perm_p_one_sided_positive": p_one_pos,
            "ci_lo": ci_lo,
            "ci_hi": ci_hi,
            "n_trials": len(sub),
            "n_subjects": n_subjects_used,
            "n_pairs_used": n_pairs_used,
            "var_type": "categorical" if categorical else "continuous",
            "n_perm": N_PERM,
            "n_boot": N_BOOT,
        })

        print(
            f"    rho={rho:.6f}, "
            f"one-sided positive p={p_one_pos:.6f}, "
            f"95% bootstrap CI=[{ci_lo:.6f}, {ci_hi:.6f}], "
            f"n_subjects={n_subjects_used}"
        )

        if len(obs_df) > 0:
            tmp = obs_df.copy()
            tmp["block"] = block_name
            tmp["model_var"] = var
            subj_detail_rows.append(tmp)

    out_df = pd.DataFrame(rows)
    out_df.to_csv(
        os.path.join(out_dir, f"rsa_{block_name}.csv"),
        index=False,
    )

    if subj_detail_rows:
        subj_detail_df = pd.concat(subj_detail_rows, ignore_index=True)
        subj_detail_df.to_csv(
            os.path.join(out_dir, f"rsa_{block_name}_subject_details.csv"),
            index=False,
        )

    return out_df


# =========================================================
# main analysis
# =========================================================
print("=== Start ACC RSA analysis: one-sided positive permutation only ===")

df = pd.read_csv(MERGED_CSV)

if "has_acc_data" in df.columns:
    df = df[df["has_acc_data"] == 1].copy()

df = (
    df
    .dropna(subset=ACC_STATE_COLS + ["Subject", "Game", "reveal_idx"])
    .copy()
    .reset_index(drop=True)
)

print(f"Input table: {MERGED_CSV}")
print(f"Output dir: {OUT_DIR}")
print(f"Data shape after filtering: {df.shape}")
print(f"Number of subjects: {df['Subject'].nunique()}")
print(f"Reveal indices available: {sorted(df['reveal_idx'].dropna().unique())}")
print(f"N_PERM={N_PERM}, N_BOOT={N_BOOT}")

# --- early pooled: reveal 1-3 ---
rsa_early = run_rsa_block_one_sided(
    df[df["reveal_idx"].isin(EARLY_WINDOW)].copy(),
    "early13",
    FOCUS_VARS,
    OUT_DIR,
)

# --- late pooled: reveal 4-6 ---
rsa_late = run_rsa_block_one_sided(
    df[df["reveal_idx"].isin(LATE_WINDOW)].copy(),
    "late46",
    FOCUS_VARS,
    OUT_DIR,
)

# =========================================================
# combine early vs late for downstream visualization
# =========================================================
if len(rsa_early) > 0:
    rsa_early = rsa_early.copy()
    rsa_early["stage"] = "Early (rev 1-3)"

if len(rsa_late) > 0:
    rsa_late = rsa_late.copy()
    rsa_late["stage"] = "Late (rev 4-6)"

compare_df = pd.concat([rsa_early, rsa_late], ignore_index=True)

compare_path = os.path.join(OUT_DIR, "rsa_early_vs_late_comparison.csv")
compare_df.to_csv(compare_path, index=False)

# 也保存一个四舍五入版本，方便快速查看
summary_path = os.path.join(OUT_DIR, "rsa_early_vs_late_comparison_rounded.csv")
compare_df.round(6).to_csv(summary_path, index=False)

print("\n=== Analysis complete ===")
print(compare_df.round(6))
print(f"\nSaved:")
print(f"- {os.path.join(OUT_DIR, 'rsa_early13.csv')}")
print(f"- {os.path.join(OUT_DIR, 'rsa_late46.csv')}")
print(f"- {compare_path}")
print(f"- {summary_path}")
print(f"- subject detail files, if available")

=== Start ACC RSA analysis: one-sided positive permutation only ===
Input table: /root/autodl-tmp/acc_rsa_results_fast/acc_rsa_merged_table_fast.csv
Output dir: /root/autodl-tmp/acc_rsa_results_final_one_sided
Data shape after filtering: (5166, 13)
Number of subjects: 10
Reveal indices available: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
N_PERM=5000, N_BOOT=2000

=== Running RSA block: early13 ===
Trials: 2583, Subjects: 10, Reveals: [np.int64(1), np.int64(2), np.int64(3)]
  - Variable: bestdeck_entropy_before | type: continuous | valid trials: 2583
    rho=0.004027, one-sided positive p=0.335933, 95% bootstrap CI=[-0.018574, 0.034547], n_subjects=10
  - Variable: uncertainty_imbalance_before | type: continuous | valid trials: 2583
    rho=-0.006230, one-sided positive p=0.836433, 95% bootstrap CI=[-0.023433, 0.010002], n_subjects=10

=== Running RSA block: late46 ===
Trials: 2583, Subjects: 10, Reveals: [np.int64(4), np.int64(5), np.int64(6)]
  - V